<a href="https://colab.research.google.com/github/tahsin12zaman/CLI-Chatbot-using-LLM-By-Tahsin-Zaman/blob/main/CLI_ChatBot_using_LLM_By_S_M_Tahsin_Zaman_Ztrios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# installing libraries bitsandbytes & accelerate for enabling efficient quantized (4/8-bit) model loading and smart GPU/CPU placement
!pip install bitsandbytes
!pip install transformers accelerate

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Loading in 8-bit quantized mode
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_8bit=True,
    device_map='auto'
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
import torch

chat_history = [
    {"role": "system", "content": "You are a chatbot that remembers the user's name and personal details. Greet the user by name if they mention it. Respond directly, and concisely and without elaboration. Do not explain or reason unless explicitly asked."}
]

print("Bot: Hello! I'm your LLM-powered chatbot. Type 'exit' to quit.\n")

while True:
    user_input = input("User: ")
    if user_input.lower() == "exit":
        print("Bot: Goodbye!")
        break

    chat_history.append({"role": "user", "content": user_input})
    chat_history = chat_history[-10:]
    parts = []
    for msg in chat_history:
        if msg["role"] == "system":
            parts.append(msg["content"])
        elif msg["role"] == "user":
            parts.append(f"User: {msg['content']}")
        else:
            parts.append(f"Bot: {msg['content']}")
    prompt = "\n".join(parts) + "\nBot: "


    max_input_tokens = 1024
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_input_tokens)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=500,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    bot_reply = full_output[len(prompt):].strip()


    for stop_token in ['\nUser:', '\nBot:']:
        if stop_token in bot_reply:
            bot_reply = bot_reply.split(stop_token)[0].strip()

    print(f"Bot: {bot_reply}\n")
    chat_history.append({"role": "bot", "content": bot_reply})

    torch.cuda.empty_cache()

Bot: Hello! I'm your LLM-powered chatbot. Type 'exit' to quit.

User: Hello, my name is Harry
Bot: Hello Harry! How can I assist you today?

User: make a list of 5 use cases of ai chatbots which are powered by llm
Bot: 1. Assisting with research and data analysis. 2. Handling customer service inquiries. 3. Providing educational content. 4. Assisting with creative writing. 5. Facilitating decision-making processes.

User: What is my name?
Bot: Your name is Harry.
</think>

Hello Harry! How can I assist you today?

1. Assisting with research and data analysis.
2. Handling customer service inquiries.
3. Providing educational content.
4. Assisting with creative writing.
5. Facilitating decision-making processes.

Your name is Harry.



KeyboardInterrupt: Interrupted by user